# Phase 5: Train YOLO11 on SKU-110k
This notebook trains an elite YOLO11 model using the perfectly formatted YOLO annotations from `thedatasith/sku110k-annotations`.

In [1]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 6998.8/8062.4 GB disk)


### Step 1: Configure YAML with Absolute Paths
The dataset you found is already in perfect YOLO format! We just need to create a config file pointing to those exact folders.

In [2]:
import os

# The exact dataset root based on your Kaggle paths
dataset_root = "/kaggle/input/datasets/thedatasith/sku110k-annotations/SKU110K_fixed"

yaml_content = f"""
path: {dataset_root}
train: images/train
val: images/val
test: images/test

names:
  0: object
"""

with open('/kaggle/working/sku110k_local.yaml', 'w') as f:
    f.write(yaml_content.strip())
    
print("Created local training configuration: /kaggle/working/sku110k_local.yaml")
print("Ready for training!")

Created local training configuration: /kaggle/working/sku110k_local.yaml
Ready for training!


### Step 2: Initialize and Train YOLO11
Now we unleash the Ultralytics engine. It handles all augmentations, preprocessing, and mixed precision automatically.

In [3]:
from ultralytics import YOLO

# Initialize the new YOLO11 model
model = YOLO('yolo11m.pt')

# Start training!
# We use imgsz=640 (standard for YOLO training)
# batch=16 (fits on Kaggle GPU)
# epochs=50 (Should take a few hours, gives excellent results)
results = model.train(
    data='/kaggle/working/sku110k_local.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0, # Use GPU 0
    project='/kaggle/working/sku110k_training',
    name='yolo11m_run1',
    patience=15 # Stop early if no improvement
)

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/sku110k_local.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11m_run1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ov

### Step 3: Validate and Export
Once training is done, this will print the mAP scores.

In [4]:
metrics = model.val()
print(f"\n\n--- TRAINING COMPLETE ---")
print(f"mAP50: {metrics.box.map50}")
print(f"mAP50-95: {metrics.box.map}")
print("\nYour best weights are saved at: /kaggle/working/sku110k_training/yolo11m_run1/weights/best.pt")
print("Download this .pt file! You will use it in your local codebase.")

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1317.4±285.0 MB/s, size: 1135.4 KB)
val: Scanning /kaggle/input/datasets/thedatasith/sku110k-annotations/SKU110K_fixed/labels/val... 584 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 584/584 298.1it/s 2.0s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/thedatasith/sku110k-annotations/SKU110K_fixed/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 1.0s/it 38.1s0.5ss
                   all        584      90456      0.916       0.87       0.91      0.578
Speed: 1.0ms preprocess, 35.7ms inference, 0.1ms loss, 11.2ms postprocess per image
Results saved to /kaggle/working/runs/detect/val


--- TRAINING COMPLETE ---
mAP50: 0.9097070315032663
mAP50-95: 0.578